# Laboratory Task 4 — PyTorch Regression

**Name:** Ara Galario  
**4th Year DS4A**

**Instruction:** Train a regression model in PyTorch using **MSE loss**, **2 fully connected layers**, **batch size 8**, **SGD**, and **1000 epochs**.

The dataset is the temperature / rainfall / humidity → apples / oranges example from the source notebook.

## 1. Imports and Data

In [7]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# inputs: temperature, rainfall, humidity
inputs = np.array([[73, 67, 43], [91, 88, 64], [87, 134, 58],
                   [102, 43, 37], [69, 96, 70]] * 3, dtype="float32")

# targets: apples, oranges
targets = np.array([[56, 70], [81, 101], [119, 133],
                    [22, 37], [103, 119]] * 3, dtype="float32")

## 2. Prepare the Dataset

Inputs and targets are standardized so SGD stays stable over 1000 epochs. Predictions are converted back to the original scale at the end.

In [8]:
x_mean, x_std = inputs.mean(0), inputs.std(0)
y_mean, y_std = targets.mean(0), targets.std(0)

X = torch.tensor((inputs - x_mean) / x_std)
Y = torch.tensor((targets - y_mean) / y_std)

dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Samples:", len(dataset))
print("Batch size:", loader.batch_size)

Samples: 15
Batch size: 8


## 3. Define the Model

Two `nn.Linear` layers are used. No activation is placed between them, so the model stays a linear regression.

In [9]:
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.Linear(4, 2)
)
print(model)

Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): Linear(in_features=4, out_features=2, bias=True)
)


## 4. Loss Function and Optimizer

In [10]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## 5. Train for 1000 Epochs

In [11]:
epochs = 1000

for epoch in range(epochs):
    total = 0
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item() * len(xb)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total / len(dataset):.6f}")

Epoch 100/1000, Loss: 0.009453
Epoch 200/1000, Loss: 0.002220
Epoch 300/1000, Loss: 0.000879
Epoch 400/1000, Loss: 0.000551
Epoch 500/1000, Loss: 0.000465
Epoch 600/1000, Loss: 0.000440
Epoch 700/1000, Loss: 0.000428
Epoch 800/1000, Loss: 0.000424
Epoch 900/1000, Loss: 0.000420
Epoch 1000/1000, Loss: 0.000416


## 6. Evaluate the Model

In [12]:
with torch.no_grad():
    pred = model(X).numpy() * y_std + y_mean   # back to original scale

mse = np.mean((pred - targets) ** 2)
print("MSE on original scale:", round(float(mse), 4))

print("\nPredictions vs. targets (first 5):")
for i in range(5):
    print(i + 1, np.round(pred[i], 2), targets[i])

MSE on original scale: 0.4922

Predictions vs. targets (first 5):
1 [56.79 70.05] [56. 70.]
2 [ 82.5  100.78] [ 81. 101.]
3 [118.63 133.13] [119. 133.]
4 [21.14 37.05] [22. 37.]
5 [101.95 119.  ] [103. 119.]


## Conclusion

The model uses all the required settings: MSE loss, two fully connected layers, batch size 8, SGD, and 1000 epochs. The last cell shows how close the predictions are to the targets.